In [0]:
df_bronze_customer_addresses = spark.read.format('parquet')\
                                        .option('header',True)\
                                        .option('inferSchema' , True)\
                                        .load("abfss://bronze@stcommercesrc.dfs.core.windows.net/customer_addresses")
display(df_bronze_customer_addresses)

address_id,customer_id,address_type,address_line1,address_line2,city,state,postal_code,country,is_default,created_at,updated_at
1,20953,HOME,"H.No. 433, Chaudry Street",,Gudivada,Maharashtra,001338,India,true,2026-02-14T09:46:12Z,2026-06-08T09:46:12Z
2,4573,OTHER,"38/63, Mall Chowk",Near Main Road,Kozhikode,Rajasthan,542351,India,true,2021-10-16T03:37:53Z,2022-08-14T03:37:53Z
3,13826,HOME,H.No. 559 Bassi Road,,Jorhat,Andhra Pradesh,618495,India,true,2025-11-29T18:29:47Z,2026-03-28T18:29:47Z
4,16560,OTHER,"10/341, Chaudhuri Ganj",,Alwar,Rajasthan,525534,India,true,2026-07-07T09:01:14Z,2027-06-04T09:01:14Z
5,22982,OTHER,"28/32, Raghavan Chowk",Near Bus Stand,Proddatur,Tamil Nadu,503056,India,false,2023-04-12T20:12:25Z,2024-02-07T20:12:25Z
6,9116,HOME,"953, Randhawa Chowk",Apartment 101,Asansol,Gujarat,238849,India,false,2024-08-24T02:20:28Z,2025-01-13T02:20:28Z
7,5095,HOME,H.No. 32 Tak Circle,2nd Floor,Nashik,Andhra Pradesh,226916,India,true,2024-03-31T15:35:46Z,2024-10-11T15:35:46Z
8,3170,OFFICE,48/01 Loke Circle,2nd Floor,Gorakhpur,West Bengal,627048,India,false,2022-07-10T01:14:59Z,2022-08-01T01:14:59Z
9,23912,OFFICE,"48/932, Edwin Path",Near Main Road,Khora,Gujarat,701543,India,true,2021-06-29T23:12:49Z,2022-04-07T23:12:49Z
10,9607,OTHER,"11/71, Dass Circle",2nd Floor,Fatehpur,West Bengal,489638,India,true,2022-09-17T22:54:37Z,2023-09-12T22:54:37Z


In [0]:
%sql
CREATE TABLE ecommerce.bronze.customer_addresses
USING PARQUET
LOCATION 'abfss://bronze@stcommercesrc.dfs.core.windows.net/customer_addresses'

In [0]:
%sql

SELECT COUNT(*) AS duplicate
FROM ecommerce.bronze.customer_addresses

duplicate
29999


**Validate AddressId**

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT address_id) AS unique_address_ids,
    COUNT(*) - COUNT(DISTINCT address_id) AS duplicate_address_ids
FROM ecommerce.bronze.customer_addresses;


total_records,unique_address_ids,duplicate_address_ids
29999,29999,0


**NULL PRIMARY KEYS**

In [0]:
%sql

SELECT COUNT(*) AS null_address_ids
FROM ecommerce.bronze.customer_addresses
WHERE address_id IS NULL;

null_address_ids
0


**Customer Referential Integrity**

In [0]:
%sql

SELECT
    ca.customer_id,
    COUNT(*) AS address_count
FROM ecommerce.bronze.customer_addresses ca
LEFT JOIN ecommerce.silver.customers c
    ON ca.customer_id = c.customer_id
WHERE c.customer_id IS NULL
GROUP BY ca.customer_id
ORDER BY address_count DESC;

customer_id,address_count
110,2
109,2
108,1
107,1


**Validate Address Fields**

In [0]:
%sql

SELECT
    COUNT(*) AS invalid_address_records
FROM ecommerce.bronze.customer_addresses
WHERE customer_id IS NULL
   OR address_type IS NULL
   OR TRIM(address_type) = ''
   OR address_line1 IS NULL
   OR TRIM(address_line1) = ''
   OR city IS NULL
   OR TRIM(city) = ''
   OR state IS NULL
   OR TRIM(state) = ''
   OR postal_code IS NULL
   OR TRIM(postal_code) = ''
   OR country IS NULL
   OR TRIM(country) = '';

invalid_address_records
0


In [0]:
%sql

SELECT
    address_type,
    COUNT(*) AS address_count
FROM ecommerce.bronze.customer_addresses
GROUP BY address_type
ORDER BY address_count DESC;

address_type,address_count
OTHER,10052
HOME,10002
OFFICE,9945


**Validate is_default**

In [0]:
%sql

SELECT
    is_default,
    COUNT(*) AS address_count
FROM ecommerce.bronze.customer_addresses
GROUP BY is_default
ORDER BY address_count DESC;

is_default,address_count
false,15088
true,14911


**Check Multiple Default Addresses**

In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS default_address_count
FROM ecommerce.bronze.customer_addresses
WHERE is_default = true
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY default_address_count DESC;

customer_id,default_address_count
1751,6
11657,5
22388,5
11594,4
6730,4
11018,4
17480,4
2744,4
5296,4
10477,4


**Measure the impact**

In [0]:
%sql

SELECT
    COUNT(*) AS customers_with_multiple_defaults,
    SUM(default_address_count) AS total_default_addresses
FROM (
    SELECT
        customer_id,
        COUNT(*) AS default_address_count
    FROM ecommerce.bronze.customer_addresses
    WHERE is_default = true
    GROUP BY customer_id
    HAVING COUNT(*) > 1
) t;

customers_with_multiple_defaults,total_default_addresses
3002,6651


**Create the Silver customer_addresses table**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.customer_addresses
USING DELTA
AS
SELECT
    address_id,
    customer_id,
    INITCAP(TRIM(address_type)) AS address_type,
    TRIM(address_line1) AS address_line1,
    TRIM(address_line2) AS address_line2,
    TRIM(city) AS city,
    INITCAP(TRIM(state)) AS state,
    TRIM(postal_code) AS postal_code,
    INITCAP(TRIM(country)) AS country,
    is_default,
    created_at,
    updated_at
FROM ecommerce.bronze.customer_addresses
WHERE address_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND address_type IS NOT NULL
  AND TRIM(address_type) <> ''
  AND address_line1 IS NOT NULL
  AND TRIM(address_line1) <> ''
  AND city IS NOT NULL
  AND TRIM(city) <> ''
  AND state IS NOT NULL
  AND TRIM(state) <> ''
  AND postal_code IS NOT NULL
  AND TRIM(postal_code) <> ''
  AND country IS NOT NULL
  AND TRIM(country) <> ''
  AND is_default IS NOT NULL
  AND created_at IS NOT NULL
  AND updated_at IS NOT NULL
  AND updated_at >= created_at;

num_affected_rows,num_inserted_rows


**Compare Bronze vs Silver**

In [0]:
%sql

SELECT
    (SELECT COUNT(*)
     FROM ecommerce.bronze.customer_addresses) AS bronze_count,

    (SELECT COUNT(*)
     FROM ecommerce.silver.customer_addresses) AS silver_count,

    (SELECT COUNT(*)
     FROM ecommerce.bronze.customer_addresses)
    -
    (SELECT COUNT(*)
     FROM ecommerce.silver.customer_addresses) AS rejected_count;

bronze_count,silver_count,rejected_count
29999,29999,0


**Create the multiple-default exception table**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.customer_address_default_exceptions
USING DELTA
AS
SELECT
    customer_id,
    COUNT(*) AS default_address_count,
    'MULTIPLE_DEFAULT_ADDRESSES' AS exception_reason,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ecommerce.bronze.customer_addresses
WHERE is_default = true
GROUP BY customer_id
HAVING COUNT(*) > 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS exception_customers
FROM ecommerce.silver.customer_address_default_exceptions;

exception_customers
3002


In [0]:
%sql

SELECT *
FROM ecommerce.silver.customer_address_default_exceptions
ORDER BY default_address_count DESC
LIMIT 10;

customer_id,default_address_count,exception_reason,validation_timestamp
1751,6,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
22388,5,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
11657,5,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
7551,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
7516,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
16934,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
360,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
8531,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
16331,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z
4403,4,MULTIPLE_DEFAULT_ADDRESSES,2026-10-03T14:33:19.353222Z


In [0]:
%sql

SELECT
    COUNT(*) AS orphan_address_records,
    COUNT(DISTINCT a.customer_id) AS orphan_customers
FROM ecommerce.bronze.customer_addresses a
LEFT JOIN ecommerce.silver.customers c
    ON a.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

orphan_address_records,orphan_customers
6,4


**Create orphan-address exception table**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.customer_address_orphan_exceptions
USING DELTA
AS
SELECT
    a.address_id,
    a.customer_id,
    'CUSTOMER_NOT_FOUND_IN_TRUSTED_SILVER' AS exception_reason,
    CURRENT_TIMESTAMP() AS validation_timestamp
FROM ecommerce.bronze.customer_addresses a
LEFT JOIN ecommerce.silver.customers c
    ON a.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS orphan_records,
    COUNT(DISTINCT customer_id) AS orphan_customers
FROM ecommerce.silver.customer_address_orphan_exceptions;

orphan_records,orphan_customers
6,4


**Create effective_default logic**

In [0]:
%sql

WITH ranked_addresses AS (
    SELECT
        address_id,
        customer_id,
        address_type,
        address_line1,
        address_line2,
        city,
        state,
        postal_code,
        country,
        is_default,
        created_at,
        updated_at,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY
                is_default DESC,
                updated_at DESC,
                address_id DESC
        ) AS default_rank

    FROM ecommerce.silver.customer_addresses
)

SELECT
    customer_id,
    address_id,
    is_default,
    updated_at,
    default_rank
FROM ranked_addresses
WHERE customer_id IN (
    SELECT customer_id
    FROM ecommerce.silver.customer_address_default_exceptions
)
ORDER BY customer_id, default_rank
LIMIT 50;

customer_id,address_id,is_default,updated_at,default_rank
13,23489,true,2025-10-28T06:42:24Z,1
13,14583,true,2023-02-14T07:13:26Z,2
39,5859,true,2024-02-27T15:27:19Z,1
39,16255,true,2021-07-30T22:27:10Z,2
41,20757,true,2024-11-05T18:51:59Z,1
41,26652,true,2023-04-17T04:03:10Z,2
49,4309,true,2024-05-14T10:50:41Z,1
49,15143,true,2022-06-07T09:58:48Z,2
67,27250,true,2022-08-09T01:59:25Z,1
67,29183,true,2022-06-30T14:21:07Z,2


**Create the final Silver Address table**

In [0]:
%sql

CREATE OR REPLACE TABLE ecommerce.silver.customer_addresses
USING DELTA
AS
WITH ranked_addresses AS (
    SELECT
        address_id,
        customer_id,
        INITCAP(TRIM(address_type)) AS address_type,
        TRIM(address_line1) AS address_line1,
        TRIM(address_line2) AS address_line2,
        TRIM(city) AS city,
        INITCAP(TRIM(state)) AS state,
        TRIM(postal_code) AS postal_code,
        INITCAP(TRIM(country)) AS country,
        is_default,
        created_at,
        updated_at,

        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY
                is_default DESC,
                updated_at DESC,
                address_id DESC
        ) AS default_rank

    FROM ecommerce.bronze.customer_addresses

    WHERE address_id IS NOT NULL
      AND customer_id IS NOT NULL
      AND address_type IS NOT NULL
      AND TRIM(address_type) <> ''
      AND address_line1 IS NOT NULL
      AND TRIM(address_line1) <> ''
      AND city IS NOT NULL
      AND TRIM(city) <> ''
      AND state IS NOT NULL
      AND TRIM(state) <> ''
      AND postal_code IS NOT NULL
      AND TRIM(postal_code) <> ''
      AND country IS NOT NULL
      AND TRIM(country) <> ''
      AND is_default IS NOT NULL
      AND created_at IS NOT NULL
      AND updated_at IS NOT NULL
      AND updated_at >= created_at
)

SELECT
    address_id,
    customer_id,
    address_type,
    address_line1,
    address_line2,
    city,
    state,
    postal_code,
    country,
    is_default,

    CASE
        WHEN default_rank = 1
             AND is_default = true
        THEN true
        ELSE false
    END AS is_effective_default,

    created_at,
    updated_at

FROM ranked_addresses;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    SUM(CASE WHEN is_effective_default = true THEN 1 ELSE 0 END)
        AS effective_default_records,
    COUNT(DISTINCT CASE
        WHEN is_effective_default = true THEN customer_id
    END) AS customers_with_effective_default
FROM ecommerce.silver.customer_addresses;

total_records,effective_default_records,customers_with_effective_default
29999,11262,11262


**Verify the "one effective default per customer" rule**

In [0]:
%sql

SELECT
    customer_id,
    COUNT(*) AS effective_default_count
FROM ecommerce.silver.customer_addresses
WHERE is_effective_default = true
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,effective_default_count


**Create the physical Silver Delta copy**

In [0]:
%sql

CREATE TABLE ecommerce.silver.customer_addresses_external
USING DELTA
LOCATION 'abfss://silver@stcommercesrc.dfs.core.windows.net/customer_addresses/'
AS
SELECT *
FROM ecommerce.silver.customer_addresses;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM ecommerce.silver.customer_addresses_external;

total_records
29999


In [0]:
%sql

DESCRIBE DETAIL ecommerce.silver.customer_addresses_external;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,ba8aa99b-e3a9-40b8-aa51-11daef4f4bcc,ecommerce.silver.customer_addresses_external,null,abfss://silver@stcommercesrc.dfs.core.windows.net/customer_addresses,2026-10-03T14:39:07.587Z,2026-10-03T14:39:08Z,List(),List(),1,1417287,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT address_id) AS unique_address_ids,
    SUM(CASE WHEN is_effective_default = true THEN 1 ELSE 0 END)
        AS effective_default_records
FROM ecommerce.silver.customer_addresses_external;

total_records,unique_address_ids,effective_default_records
29999,29999,11262
